<a href="https://colab.research.google.com/github/Le2se0hy/FA_ProAn/blob/main/RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from google.colab import drive

# ============================================================
# 1. 데이터 로드
# ============================================================
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/!Seoul_Aprtment_FINAL.xlsx'
df_raw = pd.read_excel(file_path)

# ============================================================
# 2. 전처리 함수
# ============================================================
def prepare_seoul_df_final(df_input):
    df = df_input.copy()

    # Size
    if "Size_m2" in df.columns:
        df["Size_Level"] = df["Size_m2"]

    # Population Density
    if "Pop. Density" in df.columns:
        df["Pop_Density_Level"] = df["Pop. Density"]

    # Units
    if "num_of_people" in df.columns:
        df["Units_Level"] = df["num_of_people"]

    # 비율 변수
    age_map = {
        "Median age": "Medium_age_ratio",
        "Old Population": "Old_pop_ratio",
        "Sex_ratio": "Sex_ratio_ratio"
    }

    for original, new in age_map.items():
        if original in df.columns:
            df[new] = df[original] / 100.0 if df[original].max() > 2.0 else df[original]

    # CBD 거리 km화
    if "Dist_CBD" in df.columns:
        df["Dist_CBD_km"] = df["Dist_CBD"] / 1000.0

    # 계절 더미
    if "Month_Sold" in df.columns:
        m = pd.to_numeric(df["Month_Sold"], errors="coerce")
        df["Spring"] = m.isin([3, 4, 5]).astype(int)
        df["Fall"]   = m.isin([9, 10, 11]).astype(int)
        df["Winter"] = m.isin([12, 1, 2]).astype(int)

    return df

# ============================================================
# 3. RF 실행 함수
# ============================================================
def run_random_forest_prediction(df_sub, feature_cols, target_col="Log_Price_per_m2", random_state=42):
    """
    Random Forest 회귀:
    - Train / Test 성능 비교
    - Test 예측값 저장
    """

    df_model = df_sub.copy()

    # 사용할 열만 남기고 결측 제거
    use_cols = [c for c in feature_cols + [target_col] if c in df_model.columns]
    df_model = df_model.dropna(subset=use_cols).reset_index(drop=True)

    X = df_model[[c for c in feature_cols if c in df_model.columns]]
    y = df_model[target_col]

    # train / test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=random_state
    )

    # 모델 생성
    rf = RandomForestRegressor(
        n_estimators=300,
        max_depth=15,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features='sqrt',
        random_state=random_state,
        n_jobs=-1
    )

    # 학습
    rf.fit(X_train, y_train)

    # ----------------------------
    # Train 성능
    # ----------------------------
    y_pred_train = rf.predict(X_train)
    r2_train = r2_score(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

    # ----------------------------
    # Test 성능
    # ----------------------------
    y_pred_test = rf.predict(X_test)
    r2_test = r2_score(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    # Test 예측값 테이블
    pred_df = pd.DataFrame({
        "Actual": y_test.values,
        "Predicted": y_pred_test
    }).reset_index(drop=True)

    return {
        "model": rf,
        "r2_train": r2_train,
        "rmse_train": rmse_train,
        "r2_test": r2_test,
        "rmse_test": rmse_test,
        "pred_df": pred_df,
        "n_train": len(X_train),
        "n_test": len(X_test)
    }

# ============================================================
# 4. 데이터 준비
# ============================================================
df_all = prepare_seoul_df_final(df_raw)

# 사용할 변수
features = [
    "Size_Level",
    "Floor",
    "Units_Level",
    "Parking_per_Household",
    "Construction_Year",
    "Log_Dist_Subway",
    "Log_Dist_Green",
    "Log_Dist_Water",
    "Dist_CBD_km",
    "Sex_ratio_ratio",
    "Pop_Density_Level",
    "Medium_age_ratio",
    "Old_pop_ratio",
    "Spring",
    "Fall",
    "Winter"
]

# ============================================================
# 5. 연도별 RF
# ============================================================
rf_results_by_year = []
rf_prediction_dict = {}

for year in [2022, 2023, 2024]:
    df_year = df_all[df_all["Year_Sold"] == year].copy()

    valid_cols = [c for c in features + ["Log_Price_per_m2"] if c in df_year.columns]
    df_year = df_year.dropna(subset=valid_cols)

    if len(df_year) < 30:
        print(f"{year}: 표본 수 부족으로 건너뜀")
        continue

    rf_out = run_random_forest_prediction(
        df_sub=df_year,
        feature_cols=features,
        target_col="Log_Price_per_m2",
        random_state=42
    )

    rf_results_by_year.append({
        "Year": year,
        "Train_R2": rf_out["r2_train"],
        "Train_RMSE": rf_out["rmse_train"],
        "Test_R2": rf_out["r2_test"],
        "Test_RMSE": rf_out["rmse_test"],
        "Train_N": rf_out["n_train"],
        "Test_N": rf_out["n_test"]
    })

    rf_prediction_dict[year] = rf_out["pred_df"]

    print(f"\n[{year} Random Forest 결과]")
    print("Train 성능")
    print(f"R2   : {rf_out['r2_train']:.4f}")
    print(f"RMSE : {rf_out['rmse_train']:.4f}")

    print("Test 성능")
    print(f"R2   : {rf_out['r2_test']:.4f}")
    print(f"RMSE : {rf_out['rmse_test']:.4f}")

    print(f"Train N : {rf_out['n_train']}")
    print(f"Test N  : {rf_out['n_test']}")

# ============================================================
# 6. 전체기간 RF
# ============================================================
df_full = df_all.copy()

valid_cols_full = [c for c in features + ["Log_Price_per_m2"] if c in df_full.columns]
df_full = df_full.dropna(subset=valid_cols_full)

rf_full_out = run_random_forest_prediction(
    df_sub=df_full,
    feature_cols=features,
    target_col="Log_Price_per_m2",
    random_state=42
)

print("\n[전체기간 Random Forest 결과]")
print("Train 성능")
print(f"R2   : {rf_full_out['r2_train']:.4f}")
print(f"RMSE : {rf_full_out['rmse_train']:.4f}")

print("Test 성능")
print(f"R2   : {rf_full_out['r2_test']:.4f}")
print(f"RMSE : {rf_full_out['rmse_test']:.4f}")

print(f"Train N : {rf_full_out['n_train']}")
print(f"Test N  : {rf_full_out['n_test']}")

# ============================================================
# 7. 성능표 정리
# ============================================================
rf_results_table = pd.DataFrame(rf_results_by_year)

full_row = pd.DataFrame([{
    "Year": "Full Sample",
    "Train_R2": rf_full_out["r2_train"],
    "Train_RMSE": rf_full_out["rmse_train"],
    "Test_R2": rf_full_out["r2_test"],
    "Test_RMSE": rf_full_out["rmse_test"],
    "Train_N": rf_full_out["n_train"],
    "Test_N": rf_full_out["n_test"]
}])

rf_results_table = pd.concat([rf_results_table, full_row], ignore_index=True)

print("\n[Random Forest 성능 비교표]")
display(rf_results_table)

# ============================================================
# 8. 예측값 일부 확인
# ============================================================
for year, pred_df in rf_prediction_dict.items():
    print(f"\n[{year} 예측값 일부]")
    display(pred_df.head())

print("\n[전체기간 예측값 일부]")
display(rf_full_out["pred_df"].head())

# ============================================================
# 9. 엑셀 저장
# ============================================================
output_path = '/content/drive/MyDrive/Seoul_Apartment_RF_Results.xlsx'

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    # 성능표 저장
    rf_results_table.to_excel(writer, sheet_name='RF_Performance', index=False)

    # 연도별 예측값 저장
    for year, pred_df in rf_prediction_dict.items():
        pred_df.to_excel(writer, sheet_name=f'Pred_{year}', index=False)

    # 전체기간 예측값 저장
    rf_full_out["pred_df"].to_excel(writer, sheet_name='Pred_Full', index=False)

print(f"\n저장 완료: {output_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[2022 Random Forest 결과]
Train 성능
R2   : 0.9091
RMSE : 0.1355
Test 성능
R2   : 0.8384
RMSE : 0.1861
Train N : 8463
Test N  : 2116

[2023 Random Forest 결과]
Train 성능
R2   : 0.9442
RMSE : 0.1019
Test 성능
R2   : 0.9193
RMSE : 0.1234
Train N : 24856
Test N  : 6215

[2024 Random Forest 결과]
Train 성능
R2   : 0.9583
RMSE : 0.0934
Test 성능
R2   : 0.9428
RMSE : 0.1089
Train N : 40557
Test N  : 10140

[전체기간 Random Forest 결과]
Train 성능
R2   : 0.9366
RMSE : 0.1165
Test 성능
R2   : 0.9238
RMSE : 0.1273
Train N : 129900
Test N  : 32475

[Random Forest 성능 비교표]


,Year,Train_R2,Train_RMSE,Test_R2,Test_RMSE,Train_N,Test_N
0,2022,0.909135,0.135491,0.838392,0.186131,8463,2116
1,2023,0.944247,0.101907,0.919332,0.123397,24856,6215
2,2024,0.958317,0.093422,0.942838,0.108941,40557,10140
3,Full Sample,0.936575,0.116470,0.923797,0.127331,129900,32475



[2022 예측값 일부]


,Actual,Predicted
0,17.083235,17.045568
1,15.850185,16.316097
2,16.951378,16.723207
3,16.239966,16.226575
4,16.471918,16.264183



[2023 예측값 일부]


...


[2024 예측값 일부]


,Actual,Predicted
0,16.579129,16.480600
1,16.819732,16.749400
2,16.543096,16.501551
3,15.506565,16.153138
4,16.165386,16.120198



[전체기간 예측값 일부]


,Actual,Predicted
0,16.463642,16.517932
1,16.152554,16.178088
2,16.483932,16.075550
3,16.355151,16.252175
4,16.445963,16.502992



저장 완료: /content/drive/MyDrive/Seoul_Apartment_RF_Results.xlsx
